# Robosuite Simulation — Autonomous Robot Arm

This notebook sets up **robosuite** (v1.5) for headless GPU-accelerated simulation in Google Colab.

**Environments covered:**
- `Lift` — pick up a cube
- `Stack` — stack one cube on another
- `PickPlaceSingle` — pick a block and place it in a target bin

**Requirements:** Select a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

## 1. System Setup & EGL Rendering

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

# Fix the "No private macro file found" warning
import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

print(f"robosuite {robosuite.__version__} ready (EGL rendering)")

## 2. Install Robosuite

**Important:** This cell installs robosuite then restarts the runtime to fix a numpy binary incompatibility. After the restart, **skip Cell 2** and continue from Cell 3 (environment variables).

In [ ]:
# Install robosuite, then pin numpy to 2.0.x:
#   >= 2.0  (Colab's prebuilt jax/opencv need numpy 2.x)
#   < 2.1   (numba, a robosuite dep, doesn't support numpy 2.1+)
!pip install -q robosuite imageio[ffmpeg] matplotlib
!pip install -q "numpy>=2.0,<2.1"

# Restart runtime to clear stale numpy C bindings
# After restart, SKIP this cell and continue from the next cell
import os
os.kill(os.getpid(), 9)

## 3. Helper Functions

In [ ]:
import numpy as np
import robosuite as suite
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64


def make_env(env_name="Lift", robot="Panda", camera="agentview", res=256):
    """Create a robosuite environment configured for headless rendering."""
    env = suite.make(
        env_name=env_name,
        robots=robot,
        has_renderer=False,
        has_offscreen_renderer=True,
        use_camera_obs=True,
        camera_names=camera,
        camera_heights=res,
        camera_widths=res,
        reward_shaping=True,
    )
    return env


def run_rollout(env, policy_fn=None, max_steps=300, camera="agentview"):
    """Run a rollout and collect frames + rewards.

    Args:
        env: robosuite environment
        policy_fn: callable(obs) -> action. If None, uses random actions.
        max_steps: maximum number of simulation steps
        camera: camera name for frame capture

    Returns:
        frames: list of RGB frames (H, W, 3)
        rewards: list of per-step rewards
    """
    obs = env.reset()
    frames = []
    rewards = []

    for step in range(max_steps):
        if policy_fn is not None:
            action = policy_fn(obs)
        else:
            action = np.random.uniform(-1, 1, size=env.action_dim)

        obs, reward, done, info = env.step(action)
        rewards.append(reward)

        frame = np.flip(obs[f"{camera}_image"], axis=0)
        frames.append(frame)

        if done:
            break

    return frames, rewards


def save_video(frames, path="rollout.mp4", fps=20):
    """Save a list of frames as an MP4 video."""
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()
    print(f"Video saved to {path} ({len(frames)} frames)")
    return path


def show_video(path):
    """Display an MP4 video inline in Colab."""
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512"><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
    ))


def plot_rewards(rewards, title="Reward over time"):
    """Plot per-step and cumulative rewards."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(rewards)
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Reward")
    ax1.set_title(f"{title} (per step)")

    ax2.plot(np.cumsum(rewards))
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Cumulative Reward")
    ax2.set_title(f"{title} (cumulative)")

    plt.tight_layout()
    plt.show()
    print(f"Total reward: {sum(rewards):.4f}")


print("Helper functions loaded.")

---
## 4. Experiment: Lift a Cube

The simplest manipulation task — the Panda arm must lift a cube off the table.

In [ ]:
env_lift = make_env("Lift")
print(f"Action dim: {env_lift.action_dim}")
print(f"Action spec: low={env_lift.action_spec[0]}, high={env_lift.action_spec[1]}")

frames, rewards = run_rollout(env_lift, max_steps=200)
save_video(frames, "lift_random.mp4")
show_video("lift_random.mp4")
plot_rewards(rewards, "Lift (random policy)")
env_lift.close()

---
## 5. Experiment: Stack Blocks

The robot must pick up the red cube and stack it on top of the green cube.

In [ ]:
env_stack = make_env("Stack")
print(f"Action dim: {env_stack.action_dim}")

frames, rewards = run_rollout(env_stack, max_steps=300)
save_video(frames, "stack_random.mp4")
show_video("stack_random.mp4")
plot_rewards(rewards, "Stack (random policy)")
env_stack.close()

---
## 6. Experiment: Pick and Place

The robot must pick up an object and place it in the correct bin.

In [ ]:
env_pp = make_env("PickPlaceSingle")
print(f"Action dim: {env_pp.action_dim}")

frames, rewards = run_rollout(env_pp, max_steps=400)
save_video(frames, "pickplace_random.mp4")
show_video("pickplace_random.mp4")
plot_rewards(rewards, "PickPlaceSingle (random policy)")
env_pp.close()

---
## 7. Simple Scripted Policy: Reach and Grasp

Instead of random actions, this uses a basic scripted approach to move toward the cube and close the gripper. This shows how a deliberate policy improves over random exploration.

In [ ]:
def scripted_lift_policy(obs):
    """Simple scripted policy: move toward cube, then close gripper and lift.

    Action space (7D for Panda with OSC_POSE controller):
      [dx, dy, dz, dax, day, daz, gripper]
      gripper: +1 = open, -1 = close (robosuite default for Panda)
    """
    # Get end-effector and cube positions from observations
    ee_pos = obs["robot0_eef_pos"]     # (3,) end-effector position
    cube_pos = obs["cube_pos"]         # (3,) cube position

    # Compute offset from end-effector to cube
    delta = cube_pos - ee_pos
    dist = np.linalg.norm(delta)

    action = np.zeros(7)

    if dist > 0.02:
        # Phase 1: Move toward the cube
        action[:3] = delta * 10.0  # proportional control
        action[6] = 1.0            # keep gripper open
    else:
        # Phase 2: Close gripper and lift
        action[2] = 1.0            # move up
        action[6] = -1.0           # close gripper

    return np.clip(action, -1, 1)


env_lift2 = make_env("Lift")
frames, rewards = run_rollout(env_lift2, policy_fn=scripted_lift_policy, max_steps=200)
save_video(frames, "lift_scripted.mp4")
show_video("lift_scripted.mp4")
plot_rewards(rewards, "Lift (scripted policy)")
env_lift2.close()

---
## 8. Compare Multiple Robots

Robosuite supports several robot models. Let's compare Panda vs Sawyer on the Lift task.

In [ ]:
for robot_name in ["Panda", "Sawyer", "IIWA", "Jaco"]:
    print(f"\n--- Robot: {robot_name} ---")
    try:
        env = make_env("Lift", robot=robot_name, res=128)
        frames, rewards = run_rollout(env, max_steps=100)
        save_video(frames, f"lift_{robot_name.lower()}.mp4")
        print(f"  Total reward (random, 100 steps): {sum(rewards):.4f}")
        env.close()
    except Exception as e:
        print(f"  Skipped: {e}")

---
## 9. Observation Space Explorer

Inspect what observations are available — useful for building learning policies.

In [ ]:
env = make_env("Lift")
obs = env.reset()

print("Observation keys and shapes:")
print("-" * 50)
for key, val in sorted(obs.items()):
    if isinstance(val, np.ndarray):
        print(f"  {key:30s}  shape={str(val.shape):15s}  dtype={val.dtype}")
    else:
        print(f"  {key:30s}  type={type(val).__name__}")

env.close()

---
## 10. Multi-Camera Views

Capture the scene from multiple camera angles simultaneously.

In [ ]:
cameras = ["agentview", "frontview", "sideview", "robot0_eye_in_hand"]

env_mc = suite.make(
    env_name="Stack",
    robots="Panda",
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    camera_names=cameras,
    camera_heights=256,
    camera_widths=256,
)

obs = env_mc.reset()
# Take a few random steps so the scene is more interesting
for _ in range(20):
    obs, _, _, _ = env_mc.step(np.random.uniform(-1, 1, size=env_mc.action_dim))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cam in zip(axes, cameras):
    frame = np.flip(obs[f"{cam}_image"], axis=0)
    ax.imshow(frame)
    ax.set_title(cam)
    ax.axis("off")
plt.suptitle("Multi-Camera Views — Stack Environment", fontsize=14)
plt.tight_layout()
plt.show()

env_mc.close()

---
## Available Environments Reference

| Environment | Description |
|---|---|
| `Lift` | Lift a cube off the table |
| `Stack` | Stack one cube on top of another |
| `PickPlaceSingle` | Pick up one object and place in a bin |
| `PickPlace` | Multi-object pick and place |
| `NutAssembly` | Nut-and-peg assembly (two nuts) |
| `NutAssemblySingle` | Single nut assembly |
| `Door` | Open a door with a handle |
| `Wipe` | Wipe a dirty surface |
| `TwoArmLift` | Two robots lift a pot together |
| `TwoArmPegInHole` | Two robots do peg-in-hole |
| `TwoArmHandover` | One robot hands object to the other |